# 03.2 — Machine Learning Direct Multi-Horizon Benchmark

## 1. Giới thiệu & Phạm vi
Notebook này triển khai chiến lược dự báo nhiều bước trực tiếp (**Direct Multi-Horizon Forecasting**):
$$X_t \longrightarrow [\hat{y}_{t+1}, \hat{y}_{t+2}, \dots, \hat{y}_{t+7}]$$

Các mô hình được benchmark:
- **Baseline**: `LinearRegression`
- **Tree Ensembles**: `RandomForestRegressor`, `XGBRegressor`, `LGBMRegressor`

### Tuân thủ Nghiêm ngặt các Nguyên tắc P0:
1. **P0.1 — Direct Multi-Horizon Leakage**: Dùng `build_purged_direct_samples()`. Các mẫu huấn luyện có origin $t$ sao cho $t + 7 > T_{train}$ sẽ bị loại bỏ hoàn toàn (Purged Boundary).
2. **P0.3 — Chính sách Future Covariates**: Chỉ sử dụng các đặc trưng lịch sử ($t, t-1, \dots$) từ `FeatureBuilder` và các đặc trưng lịch đã biết trước (tháng, ngày trong năm, sin/cos). Tuyệt đối không dùng biến thời tiết tương lai thực tế.

Toàn bộ kết quả được xuất ra:
- `data/processed/results/ml_direct_results.csv`
- `data/processed/results/ml_direct_forecasts.npz`


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    load_modeling_contract,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor

from src.featurengineering.builder import FeatureBuilder, FeatureConfig

contract = load_modeling_contract()
train_full, val_set, test_set = load_canonical_data()
print(f"Loaded canonical splits. Horizon H={contract['forecast_horizon']}")


Loaded canonical splits. Horizon H=7


## 2. Feature Pipeline (Chỉ Fit trên Tập Train) & Chống Rò rỉ Lịch sử

In [2]:
fb_config = FeatureConfig(
    lag_columns=["Lượng mưa", "Nhiệt độ 2m", "Độ ẩm tương đối 2m", "Tốc độ gió 2m"],
    lag_periods=[1, 2, 3, 7],
    rolling_columns=["Lượng mưa", "Nhiệt độ 2m"],
    rolling_windows=[7, 14, 30],
    rolling_stats=["mean", "std"],
    temporal_features=["Month_sin", "Month_cos", "DayOfYear_sin", "DayOfYear_cos", "Is_Wet_Season"],
)

fb = FeatureBuilder(fb_config)
fb.fit(train_full)

train_feat = fb.transform(train_full)
test_feat = fb.transform(test_set)

feature_cols = [c for c in train_feat.columns if c not in ['Ngày', 'Lượng mưa']]
print(f"Constructed {len(feature_cols)} leakage-free features.")


FeatureBuilder.fit() - learning from train data only
[STATS] fit_missing_value_stats: computed fill_mean for 30 columns (train only)
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 2m', 'Độ ẩm tương đối 2m', 'Tốc độ gió 2m']
   Lag periods: [1, 2, 3, 7]
   Dataset shape: (7426, 47)
[ROLLING] CREATING ROLLING FEATURES
   [LEAK-SAFE] Leakage-safe mode: shift(1) applied before rolling
   [OK] Created 12 rolling features
   Columns: ['Lượng mưa', 'Nhiệt độ 2m']
   Windows: [7, 14, 30]
   Statistics: ['mean', 'std']
   Dataset shape: (7426, 59)
[TEMPORAL] CREATING TEMPORAL FEATURES
   [OK] Created 5 temporal features
   Dataset shape: (7426, 64)
[STATS] fit_feature_quality: 62 features analyzed (train only)
   Keep: 55, Drop: 7
   FeatureBuilder fitted
[FEATURE-APPLY] apply_missing_value_fill: filled 0 values using train-derived stats
[LAG] CREATING LAG FEATURES
   [OK] Created 16 lag features
   Columns lagged: ['Lượng mưa', 'Nhiệt độ 

## 3. Tạo Mẫu Purged Multi-Horizon (P0.1 Boundary Leakage Prevention)

In [3]:
horizon = contract['forecast_horizon']

X_train, Y_train, train_dates = build_purged_direct_samples(train_feat, feature_cols, horizon=horizon)
X_test, Y_test, test_dates = build_purged_direct_samples(test_feat, feature_cols, horizon=horizon)

print(f"X_train shape: {X_train.shape}, Y_train shape: {Y_train.shape}")
print(f"X_test shape : {X_test.shape}, Y_test shape : {Y_test.shape}")
assert Y_train.shape[1] == horizon, "Multi-horizon target width mismatch!"


X_train shape: (7389, 55), Y_train shape: (7389, 7)
X_test shape : (2154, 55), Y_test shape : (2154, 7)


## 4. Huấn luyện & Đánh giá Nhóm ML Direct

In [4]:
models = {
    'LinearRegression': MultiOutputRegressor(LinearRegression()),
    'RandomForest': RandomForestRegressor(n_estimators=80, max_depth=8, n_jobs=-1, random_state=42),
    'XGBoost_Direct': MultiOutputRegressor(XGBRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42
    )),
    'LightGBM_Direct': MultiOutputRegressor(LGBMRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.03, num_leaves=31, subsample=0.8, colsample_bytree=0.8, verbosity=-1, random_state=42
    ))
}

results_list = []
forecasts_dict = {}

sub_indices = np.linspace(0, len(X_test) - 1, min(50, len(X_test)), dtype=int)
X_test_eval = X_test[sub_indices]
Y_test_eval = Y_test[sub_indices]

for name, model in models.items():
    print(f"Training {name}...")
    t0 = time.time()
    model.fit(X_train, Y_train)
    t_train = time.time() - t0
    
    t0 = time.time()
    y_pred = model.predict(X_test_eval)
    t_infer = time.time() - t0
    
    y_pred = np.clip(y_pred, 0, None)
    forecasts_dict[name] = y_pred
    
    metrics = compute_benchmark_metrics(
        y_true=Y_test_eval,
        y_pred=y_pred,
        model_name=name,
        group='Machine Learning',
        strategy='ML Direct Baseline' if name == 'LinearRegression' else 'Direct Multi-Horizon Tree',
        training_time=t_train,
        inference_time=t_infer,
        parameter_count=1000 if 'Forest' in name or 'Boost' in name or 'LGBM' in name else 100,
        rain_threshold=contract['rain_threshold']
    )
    results_list.append(metrics)

df_results = pd.concat(results_list, ignore_index=True).sort_values(by='CR-MAE@7')
csv_path, npz_path = save_benchmark_artifacts(df_results, forecasts_dict, Y_test_eval, "ml_direct")
print(f">> Saved artifacts to:\n  - {csv_path}\n  - {npz_path}\n")

print("=== ML DIRECT BENCHMARK RESULTS (Ranked by CR-MAE@7) ===")
display(df_results[['model', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7']])


Training LinearRegression...
Training RandomForest...
Training XGBoost_Direct...
Training LightGBM_Direct...
>> Saved artifacts to:
  - data\processed\results\ml_direct_results.csv
  - data\processed\results\ml_direct_forecasts.npz

=== ML DIRECT BENCHMARK RESULTS (Ranked by CR-MAE@7) ===
              model                   strategy  ...  CR-MAE@7  CR-Bias@7
0  LinearRegression         ML Direct Baseline  ...   17.7864    -7.1560
3   LightGBM_Direct  Direct Multi-Horizon Tree  ...   18.3126    -4.5106
1      RandomForest  Direct Multi-Horizon Tree  ...   18.3630    -5.5001
2    XGBoost_Direct  Direct Multi-Horizon Tree  ...   19.7954    -2.5995

[4 rows x 7 columns]
